# 🏥 Sistema de Diagnóstico de Doenças - Análise Exploratória

**Projeto Final INAR - Terceiro Ano, II Semestre**

Este notebook demonstra como usar o sistema de diagnóstico de doenças desenvolvido, incluindo:
- Carregamento e validação de dados
- Análise exploratória detalhada
- Pré-processamento de dados
- Treinamento de modelos
- Avaliação e comparação de resultados

⚠️ **Aviso**: Este sistema é apenas para fins educacionais.

## 📚 Importações e Configuração Inicial

In [ ]:
# Importações básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

# Adicionar paths dos módulos
import sys
import os
sys.path.append(os.path.join('..', 'src', 'utils'))
sys.path.append(os.path.join('..', 'src', 'models'))

print("📦 Bibliotecas importadas com sucesso!")

## 📂 Carregamento de Dados

In [ ]:
# Importar nossos módulos
from carregador_dados import CarregadorDados

# Caminho do dataset
caminho_dataset = '../dataset/Disease_symptom_and_patient_profile_dataset.csv'

# Criar carregador e carregar dados
carregador = CarregadorDados(caminho_dataset)
dados = carregador.carregar_dados()

# Validar estrutura
if carregador.validar_estrutura():
    print("✅ Estrutura do dataset validada!")
else:
    print("❌ Problema na estrutura do dataset")

# Exibir informações básicas
info = carregador.obter_informacoes_basicas()
print(f"\n📊 Informações do Dataset:")
print(f"• Registros: {info['numero_linhas']:,}")
print(f"• Colunas: {info['numero_colunas']}")
print(f"• Doenças únicas: {info['doencas_unicas']}")
print(f"• Uso de memória: {info['memoria_uso']:.2f} MB")

In [ ]:
# Visualizar primeiras linhas
print("🔍 Primeiras 5 linhas do dataset:")
dados.head()

In [ ]:
# Informações sobre tipos de dados e valores nulos
print("📋 Informações sobre o dataset:")
dados.info()

## 🔍 Análise Exploratória Detalhada

In [ ]:
# Importar analisador exploratório
from analise_exploratoria import AnalisadorExploratorio

# Criar analisador
analisador = AnalisadorExploratorio(dados)

# Obter estatísticas descritivas
stats = analisador.estatisticas_descritivas()

print("📊 ESTATÍSTICAS DESCRITIVAS:")
print("=" * 40)

# Estatísticas gerais
print(f"Total de registros: {stats['geral']['total_registros']:,}")
print(f"Total de colunas: {stats['geral']['total_colunas']}")

# Distribuição de classes
if 'outcome' in stats:
    print("\n🎯 Distribuição de Classes (Outcome):")
    for classe, count in stats['outcome'].items():
        perc = stats['outcome_percentual'][classe]
        print(f"  • {classe}: {count:,} ({perc:.1f}%)")

# Estatísticas demográficas
if 'idade' in stats:
    print(f"\n👥 Idade dos Pacientes:")
    print(f"  • Média: {stats['idade']['media']:.1f} anos")
    print(f"  • Mediana: {stats['idade']['mediana']:.1f} anos")
    print(f"  • Faixa: {stats['idade']['min']:.0f} - {stats['idade']['max']:.0f} anos")

if 'genero' in stats:
    print(f"\n⚧ Distribuição por Gênero:")
    for genero, count in stats['genero'].items():
        print(f"  • {genero}: {count:,}")

In [ ]:
# Visualização da distribuição de classes
analisador.visualizar_distribuicao_classes()

In [ ]:
# Visualização das doenças mais comuns
analisador.visualizar_distribuicao_doencas(top_n=15)

In [ ]:
# Visualização dos sintomas
analisador.visualizar_sintomas()

In [ ]:
# Análise demográfica
analisador.analise_demografica()

In [ ]:
# Matriz de correlação entre sintomas
analisador.visualizar_correlacao_sintomas()

## 🔧 Pré-processamento de Dados

In [ ]:
# Importar preprocessador
from preprocessador import PreProcessadorDados

# Criar preprocessador
preprocessador = PreProcessadorDados(dados)

# Executar pré-processamento completo
print("🔧 Iniciando pré-processamento...")
dados_processados = preprocessador.processar_dados_completo(test_size=0.2, random_state=42)

print("\n✅ Pré-processamento concluído!")
print(f"• Features de treino: {dados_processados['X_train'].shape}")
print(f"• Features de teste: {dados_processados['X_test'].shape}")
print(f"• Target de treino: {dados_processados['y_train'].shape}")
print(f"• Target de teste: {dados_processados['y_test'].shape}")

print(f"\n📋 Features utilizadas:")
for i, feature in enumerate(dados_processados['feature_names'], 1):
    print(f"  {i:2d}. {feature}")

## 🤖 Treinamento de Modelos

In [ ]:
# Importar treinador de modelos
from treinador_modelos import TreinadorModelos

# Criar treinador
treinador = TreinadorModelos()

print("🚀 Iniciando treinamento de modelos...")
print("⏱️ Isso pode levar alguns minutos...")

# Treinar todos os modelos (sem grid search para rapidez)
resultados_treino = treinador.treinar_todos_modelos(
    dados_processados['X_train'],
    dados_processados['y_train'],
    usar_grid_search=False,  # Para rapidez no notebook
    cv_folds=3
)

print("\n✅ Treinamento concluído!")
print(f"🏆 Melhor modelo: {treinador._obter_nome_melhor_modelo()}")
print(f"📊 Score: {treinador.melhor_score:.4f}")

In [ ]:
# Obter ranking dos modelos
ranking = treinador.obter_ranking_modelos(
    dados_processados['X_test'],
    dados_processados['y_test']
)

print("🏆 RANKING DOS MODELOS:")
print("=" * 80)
print(ranking)

## 📊 Avaliação Detalhada

In [ ]:
# Importar avaliador
from avaliador import AvaliadorModelos

# Criar avaliador
avaliador = AvaliadorModelos()

# Avaliar o melhor modelo em detalhes
melhor_modelo = treinador.melhor_modelo
nome_melhor_modelo = treinador._obter_nome_melhor_modelo()

print(f"🔍 Avaliação detalhada do melhor modelo: {nome_melhor_modelo}")
resultado_avaliacao = avaliador.avaliar_modelo_completo(
    melhor_modelo,
    dados_processados['X_test'],
    dados_processados['y_test'],
    nome_melhor_modelo
)

In [ ]:
# Comparar todos os modelos
resultados_todos = {}
for nome_modelo, modelo in treinador.modelos_treinados.items():
    resultado = avaliador.avaliar_modelo_completo(
        modelo,
        dados_processados['X_test'],
        dados_processados['y_test'],
        nome_modelo
    )
    resultados_todos[nome_modelo] = resultado

# Comparação visual
avaliador.comparar_modelos(resultados_todos)

## 🔮 Teste do Sistema de Predição

In [ ]:
# Salvar modelo e preprocessador para teste
import joblib
import os

# Criar pasta de modelos
pasta_modelos = '../modelos_salvos'
os.makedirs(pasta_modelos, exist_ok=True)

# Salvar melhor modelo
joblib.dump(melhor_modelo, os.path.join(pasta_modelos, 'melhor_modelo.pkl'))
joblib.dump({'nome': nome_melhor_modelo, 'score': treinador.melhor_score}, 
           os.path.join(pasta_modelos, 'info_melhor_modelo.pkl'))

# Salvar preprocessador
preprocessador.salvar_preprocessadores(pasta_modelos)

print("💾 Modelo e preprocessador salvos!")

In [ ]:
# Importar sistema de predição
from sistema_predicao import SistemaPredicao

# Criar sistema de predição
sistema = SistemaPredicao(pasta_modelos)

# Verificar status
status = sistema.status_sistema()
print(f"📋 Status do sistema: {status}")

if status['modelo_carregado']:
    print("✅ Sistema pronto para predições!")
else:
    print("❌ Problema ao carregar o sistema")

In [ ]:
# Testar predição com dados de exemplo
dados_paciente_1 = {
    'febre': True,
    'tosse': False,
    'fadiga': True,
    'dificuldade_respirar': False,
    'idade': 35,
    'genero': 'Male',
    'pressao': 'Normal',
    'colesterol': 'Normal'
}

# Fazer predição
resultado = sistema.fazer_predicao(dados_paciente_1)

print("🔍 RESULTADO DA PREDIÇÃO - PACIENTE 1:")
print("=" * 50)
print(f"🎯 Resultado: {resultado['resultado']} {resultado['resultado_interpretado']['emoji']}")
print(f"📊 Confiança: {resultado['confianca']:.2%}")
print(f"📋 Status: {resultado['resultado_interpretado']['status']}")
print(f"💡 Recomendação: {resultado['resultado_interpretado']['recomendacao']}")

if resultado['probabilidades']:
    print(f"\n📈 Probabilidades:")
    print(f"  • Negativo: {resultado['probabilidades']['negativo']:.2%}")
    print(f"  • Positivo: {resultado['probabilidades']['positivo']:.2%}")

In [ ]:
# Testar com outro paciente
dados_paciente_2 = {
    'febre': False,
    'tosse': True,
    'fadiga': False,
    'dificuldade_respirar': False,
    'idade': 28,
    'genero': 'Female',
    'pressao': 'Low',
    'colesterol': 'High'
}

resultado_2 = sistema.fazer_predicao(dados_paciente_2)

print("🔍 RESULTADO DA PREDIÇÃO - PACIENTE 2:")
print("=" * 50)
print(f"🎯 Resultado: {resultado_2['resultado']} {resultado_2['resultado_interpretado']['emoji']}")
print(f"📊 Confiança: {resultado_2['confianca']:.2%}")
print(f"📋 Status: {resultado_2['resultado_interpretado']['status']}")
print(f"💡 Recomendação: {resultado_2['resultado_interpretado']['recomendacao']}")

In [ ]:
# Visualizar histórico de predições
historico = sistema.obter_historico()
stats_historico = sistema.obter_estatisticas_historico()

print("📊 ESTATÍSTICAS DO HISTÓRICO:")
print("=" * 40)
print(f"• Total de predições: {stats_historico['total_predicoes']}")
print(f"• Predições positivas: {stats_historico['predicoes_positivas']}")
print(f"• Predições negativas: {stats_historico['predicoes_negativas']}")
print(f"• Confiança média: {stats_historico['confianca_media']:.2%}")

if stats_historico['total_predicoes'] > 0:
    print(f"• % Positivos: {stats_historico['percentual_positivos']:.1f}%")
    print(f"• % Negativos: {stats_historico['percentual_negativos']:.1f}%")

## 📋 Relatório Final

In [ ]:
# Gerar relatório final comparativo
relatorio_final = avaliador.gerar_relatorio_final(resultados_todos)

print("\n🎉 ANÁLISE CONCLUÍDA COM SUCESSO!")
print("=" * 60)
print(f"✅ Dataset analisado: {dados.shape[0]:,} registros")
print(f"🤖 Modelos treinados: {len(treinador.modelos_treinados)}")
print(f"🏆 Melhor modelo: {nome_melhor_modelo}")
print(f"📊 Melhor score: {treinador.melhor_score:.4f}")
print(f"🔮 Predições testadas: {stats_historico['total_predicoes']}")

print("\n💡 PRÓXIMOS PASSOS:")
print("• Use a interface gráfica: python main.py --gui")
print("• Execute o pipeline completo: python main.py --pipeline")
print("• Teste com seus próprios dados")

print("\n⚠️  LEMBRETE IMPORTANTE:")
print("Este sistema é apenas para fins educacionais.")
print("Para questões médicas reais, consulte um profissional de saúde.")